# Forward Pass Debugging: Layer-by-Layer Reference

This notebook performs a forward pass through a .keras model one layer at a time, for the purpose of verifying low-level implementations in C or RISC-V.

At each step, it prints the output of the current layer. These values act as ground truth for validating manual implementations. The output of one layer is passed directly as the input to the next, preserving the inference flow.

This setup helps identify discrepancies between the high-level model and its low-level counterparts, allowing for precise, layer-specific debugging.


In [3]:
import os
import tensorflow as tf
import numpy as np
import random

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

## 1.0 Load and preprocess the data

In [4]:
(images, labels), _ = tf.keras.datasets.mnist.load_data()
images = images.astype("float32") / 255.0  # Normalize the images to [0, 1]
images = np.expand_dims(images, -1)  # Add channel dimension
labels = tf.keras.utils.to_categorical(labels, 10)  # One-hot encode the labels

## 2.0 Helper Functions

### 2.1 Function to Get a random image and label

Get a random image and label from the dataset. This function is used to generate a random input for the model.

In [5]:
def get_random_image():
    # Randomly select an image and its label
    index = random.randint(0, len(images) - 1)
    image = images[index].squeeze()  # (28, 28)
    image = tf.expand_dims(image, axis=0)  # Add batch dimension (1, 28, 28)
    image = tf.expand_dims(image, axis=-1)  # Add channel dimension (1, 28, 28, 1)
    label = np.argmax(labels[index])  # Get label
    return image, label

### 2.2 Function to Print a tensor

Prints the shape and values of a tensor in a readable format.

For 4D tensors (e.g., batches of images), it prints the values of the first sample,
channel by channel. For 2D tensors (e.g., dense layer outputs), it prints all values
row by row. Other shapes are not currently supported.

In [6]:
def print_output_shape_and_values(x):
    print(f"Output shape: {x.shape}")
    
    # If it's a 4D tensor (e.g., batch of images), handle it
    if len(x.shape) == 4:
        _, height, width, channels = x.shape
        for c in range(channels):
            for i in range(height):
                for j in range(width):
                    print(f"{x[0, i, j, c]:.3f}", end=" ")
                print()
            print()
    
    # If it's a 2D array (after flattening), handle it
    elif len(x.shape) == 2:
        rows, cols = x.shape
        for i in range(rows):
            for j in range(cols):
                print(f"{x[i, j]:.3f}", end=" ")
            print()
    else:
        print("Unsupported shape")


## 3.0 Load the model from mnist_cnn_model.keras

In [9]:
model = tf.keras.models.load_model("../models/mnist_cnn_model.keras")

## 4.0 Get a random image, label and step through the model layer by layer

### 4.1 Get a random image and label

In [10]:
image, label = get_random_image()
print(f"Label: {label}")

Label: 7


### 4.2 Step through the model layer by layer

#### 4.2.1 Input Image

In [11]:
print("Original Image:")
print_output_shape_and_values(image)

Original Image:
Output shape: (1, 28, 28, 1)
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.

#### 4.2.2 Conv2D Layer

Note: Refer to section **4.2.1** for the input

In [12]:
conv2d_out = model.layers[0](image)
print_output_shape_and_values(conv2d_out)

Output shape: (1, 24, 24, 8)
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 
-0.170 -0.036 -0.095 -0.107 -0.277 -0.384 -0.358 -0.357 -0.322 -0.317 -0.317 -0.317 -0.397 -0.504 -0.508 -0.498 -0.473 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 
0.054 -0.003 0.166 0.295 0.386 0.388 0.326 0.217 0.130 0.120 0.120 0.120 0.093 0.080 -0.043 -0.250 -0.351 -0.425 -0.504 -0.510 -0.495 -0.477 -0.460 -0.460 
-0.171 -0.201 0.028 0.251 0.332 0.425 0.445 0.381 0.350 0.347 0.347 0.347 0.389 0.472 0.484 0.424 0.338 0.203 -0.095 -0.359 -0.424 -0.500 -

#### 4.2.3 ReLU Activation

Note: Refer to section **4.2.2** for the input

In [13]:
relu_out = model.layers[1](conv2d_out)
print_output_shape_and_values(relu_out)

Output shape: (1, 24, 24, 8)
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.054 0.000 0.166 0.295 0.386 0.388 0.326 0.217 0.130 0.120 0.120 0.120 0.093 0.080 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.028 0.251 0.332 0.425 0.445 0.381 0.350 0.347 0.347 0.347 0.389 0.472 0.484 0.424 0.338 0.203 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.136 0.284 0.314 0.351 0.425

#### 4.2.4 MaxPooling

Note: Refer to section **4.2.3** for the input

In [14]:
maxpool_out = model.layers[2](relu_out)
print_output_shape_and_values(maxpool_out)

Output shape: (1, 12, 12, 8)
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.054 0.295 0.425 0.445 0.350 0.347 0.472 0.484 0.338 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.284 0.351 0.434 0.287 0.005 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.042 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.203 0.084 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.186 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.151 0.020 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.206 0.191 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 

0.178 0.178 0.178 0.178 0.178 0.178 0.178 0.178 0.178 0.178 0.178 0.178 
0.910 1.551 1.763 1.4

#### 4.2.5 Flatten

Note: Refer to section **4.2.4** for the input

In [15]:
flatten_out = model.layers[3](maxpool_out)
print_output_shape_and_values(flatten_out)

Output shape: (1, 1152)
0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.910 0.000 0.000 0.137 0.127 0.449 0.000 0.000 1.551 0.000 0.000 0.500 0.000 0.592 0.000 0.000 1.763 0.000 0.000 0.000 0.000 0.020 0.000 0.000 1.473 0.000 0.000 0.000 0.000 0.000 0.000 0.000 1.062 0.000 0.000 0.000 0.000 0.000 0.000 0.000 1.034 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.921 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.624 0.000 0.090 0.000 0.000 0.000 0.000 0.000 0.255 0.00

#### 4.2.6 Fifth Layer: Dense Layer

Note: Refer to section **4.2.5** for the input

In [16]:
dense_out = model.layers[4](flatten_out)  # Fifth layer output
print_output_shape_and_values(dense_out)

Output shape: (1, 10)
-7.046 -5.199 -0.689 -4.105 -15.555 -10.419 -31.470 14.884 -9.137 -6.172 


#### 4.2.7 Layer Six: Softmax

Note: Refer to section **4.2.6** for the input

In [17]:
softmax_out = model.layers[5](dense_out)
print_output_shape_and_values(softmax_out)

Output shape: (1, 10)
0.000 0.000 0.000 0.000 0.000 0.000 0.000 1.000 0.000 0.000 


### 4.3 Get model prediction

In [18]:
print(f"Predicted class: {np.argmax(softmax_out)}")
print(f"True class: {label}")

Predicted class: 7
True class: 7
